# Voltage and current need different convergence bounds

This notebook exercises the opt-in discipline API. A node voltage has a current-balance residual; a voltage source's current state has a voltage constraint. We inspect both, solve picoampere loads, and control transient local error separately. Run with the Circulax development environment and x64 enabled.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from circulax import Nature, Discipline, TolerancePolicy, compile_circuit
from circulax.components.base_component import component
from circulax.components.electronic import VoltageSource, Resistor, Capacitor

net = {
    "instances": {"GND": {"component": "ground"},
                  "V1": {"component": "v", "settings": {"V": 1.0}},
                  "R1": {"component": "r", "settings": {"R": 1e12}}},
    "connections": {"V1,p1": "R1,p1", "V1,p2": "GND,p1", "R1,p2": "GND,p1"},
}
models = {"ground": lambda: 0, "v": VoltageSource, "r": Resistor}
policy = TolerancePolicy(atol={"current": 1e-14})
circuit = compile_circuit(net, models, backend="dense", tolerances=policy)
layout = circuit.convergence_layout
for name, unknown, equation, xa, ra in zip(layout.names, layout.unknown_natures, layout.equation_natures,
                                         layout.unknown_atol, layout.residual_atol):
    if unknown is not None:
        print(f"{name:12s} unknown={unknown.units:4s} atol={float(xa):.1e}; equation={equation.units:4s} atol={float(ra):.1e}")


## A scalar absolute bound misses the distinction

For an illustrative 1 nA correction near zero, a scalar absolute tolerance of 1e-6 accepts the update. A 1e-14 A current bound rejects it. This is a comparison of convergence criteria, not a claim that this linear example produces an incorrect legacy solution.

In [ ]:
correction = 1e-9
print("Legacy normalized update:", correction / 1e-6)
print("Current normalized update:", correction / 1e-14)
y, converged = circuit.dc_checked()
assert converged
assert np.isclose(y[circuit.port_map["V1,i_src"]], -1e-12, atol=1e-15, rtol=0)
report = circuit.dc_diagnostics(y)
print("Source current [A]:", float(y[circuit.port_map["V1,i_src"]]))
print("Normalized equation residual:", float(report["residual_error"]))
assert report["residual_error"] < 1


In [ ]:
resistances = jnp.logspace(3, 12, 30)
ys = circuit.dc(R=resistances)
source_current = -ys[:, circuit.port_map["V1,i_src"]]
np.testing.assert_allclose(source_current, 1 / resistances, rtol=1e-6, atol=1e-15)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(resistances, source_current, "o", label="Circulax")
ax.loglog(resistances, 1 / resistances, "--", label="1 V / R")
ax.set(xlabel="Resistance [ohm]", ylabel="Load current [A]")
ax.legend(); ax.grid(True, which="both", alpha=.3)
plt.show()


## Transient local error has its own budget

We charge a 1 uF capacitor through 1 kohm. The source steps to 1 V at t=0, with a discharged capacitor and consistent algebraic source values. `local_error_factor=10` loosens the adaptive integration budget relative to nonlinear convergence. The analytic curve checks waveform accuracy independently of Newton convergence.

In [ ]:
rc_net = {
    "instances": {"GND": {"component": "ground"}, "V1": {"component": "v", "settings": {"V": 1.0}},
                  "R1": {"component": "r", "settings": {"R": 1e3}},
                  "C1": {"component": "c", "settings": {"C": 1e-6}}},
    "connections": {"V1,p1": "R1,p1", "R1,p2": "C1,p1", "V1,p2": "GND,p1", "C1,p2": "GND,p1"},
}
rc = compile_circuit(rc_net, {**models, "c": Capacitor}, backend="dense",
                     tolerances=TolerancePolicy(local_error_factor=10))
ts = jnp.linspace(0, 0.005, 101)
y0 = jnp.zeros(rc.sys_size).at[rc.port_map["V1,p1"]].set(1.0).at[rc.port_map["V1,i_src"]].set(-0.001)
solution = rc.transient(t0=0, t1=0.005, dt0=1e-5, y0=y0, saveat=ts,
                        stepsize_controller=rc.adaptive_controller(), throw=True, max_steps=10000)
v = solution.ys[:, rc.port_map["C1,p1"]]
reference = 1 - np.exp(-np.asarray(ts) / 1e-3)
assert np.max(np.abs(v - reference)) < 3e-3
print("Maximum waveform error [V]:", float(np.max(np.abs(v - reference))))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ts * 1e3, v, label="Circulax")
ax.plot(ts * 1e3, reference, "--", label="Analytic RC")
ax.set(xlabel="Time [ms]", ylabel="Capacitor voltage [V]")
ax.legend(); ax.grid(alpha=.3)
plt.show()


## Migration

Omit `tolerances=` to keep the scalar API. Do not combine a policy with scalar `atol` or `rtol`. Physical mode requires model metadata and checks stronger convergence conditions; it can reject an estimate previously accepted by the scalar interface. See `docs/disciplines.md` for coverage and remaining work.